# 1

## lib and config

In [1]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from PIL import Image
import torchvision.transforms.functional as TF
import matplotlib.pyplot as plt
from dataclasses import dataclass

In [2]:
# ==========================================
# 1. 配置参数 (Config)
# ==========================================
cuda_num = 0
@dataclass
class Config:
    DATASET_DIR: str = "/home/yzm"
    IMG_SIZE: int = 128
    BATCH_SIZE: int = 8
    BASE_LR: float = 1e-2          # ② 称为 base_lr
    EPOCHS: int = 50
    DEVICE: torch.device = torch.device(f"cuda:{cuda_num}" if torch.cuda.is_available() else "cpu")
    LABELED_RATIO: float = 0.10
    UNLABELED_RATIO: float = 0.90 # 无标签
    CONSISTENCY_WEIGHT: float = 0.01
    CONSISTENCY_RAMPUP: float = 200.0 # ② 的 rampup 步数
    EMA_ALPHA: float = 0.99
    NUM_CLASSES: int = 3

config = Config()

## Dice

In [3]:
class DiceLoss(nn.Module):
    def __init__(self, n_classes):
        super(DiceLoss, self).__init__()
        self.n_classes = n_classes

    def _one_hot_encoder(self, input_tensor):
        tensor_list = []
        for i in range(self.n_classes):
            temp_prob = input_tensor == i
            tensor_list.append(temp_prob.unsqueeze(1))
        output_tensor = torch.cat(tensor_list, dim=1)
        return output_tensor.float()

    def _dice_loss(self, score, target):
        target = target.float()
        smooth = 1e-5
        intersect = torch.sum(score * target)
        y_sum = torch.sum(target * target)
        z_sum = torch.sum(score * score)
        loss = (2 * intersect + smooth) / (z_sum + y_sum + smooth)
        return 1 - loss

    def forward(self, inputs, target, weight=None, softmax=False):
        if softmax:
            inputs = torch.softmax(inputs, dim=1)
        target = self._one_hot_encoder(target)
        if weight is None:
            weight = [1.0] * self.n_classes
        assert inputs.size() == target.size(), 'predict & target shape do not match'
        class_wise_dice = []
        loss = 0.0
        for i in range(0, self.n_classes):
            dice = self._dice_loss(inputs[:, i], target[:, i])
            class_wise_dice.append(dice)
            loss += dice * weight[i]
        return loss / self.n_classes

## unet

In [4]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.double_conv(x)

class Down(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(nn.MaxPool2d(2), DoubleConv(in_channels, out_channels))
    def forward(self, x): return self.maxpool_conv(x)

class Up(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
        self.conv = DoubleConv(in_channels, out_channels)
    def forward(self, x1, x2):
        x1 = self.up(x1)
        diffY, diffX = x2.size()[2] - x1.size()[2], x2.size()[3] - x1.size()[3]
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

In [5]:
class UNet(nn.Module):
    def __init__(self, n_channels=1, n_classes=3):
        super(UNet, self).__init__()
        self.inc = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        # 关键修正：Bottleneck 层应该输出 1024 通道
        self.down4 = Down(512, 1024) 
        
        # 关键修正：对应 Up 层的输入输出通道
        self.up1 = Up(1024, 512)
        self.up2 = Up(512, 256)
        self.up3 = Up(256, 128)
        self.up4 = Up(128, 64)
        self.outc = nn.Conv2d(64, n_classes, kernel_size=1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        # 逐层上采样并拼接
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        return self.outc(x)

## data

In [6]:
class Target_ACDC(Dataset):
    def __init__(self, phase='train', patient_indices=None):
        """
        Args:
            phase: 'train', 'val', 'test'
            patient_indices: 指定使用哪些病人的索引（用于划分标注/无标注数据）
        """
        self.root_dir = os.path.join(config.DATASET_DIR, "ACDC")
        self.target_size = config.IMG_SIZE
        
        # 1. 先找到所有病人的 npy 文件（在 {phase}_image 目录下，不进入 slices）
        all_patient_files = []
        group_path = [entry for entry in os.scandir(self.root_dir) if entry.is_dir()]
        
        for gp in group_path:
            for frame_path in os.scandir(gp):
                if frame_path.name not in ['ED', 'ES']: 
                    continue
                # 【关键修改】直接在 train_image/val_image/test_image 目录找 npy，不进入 slices
                p_path = os.path.join(frame_path.path, f'{phase}_image')
                if not os.path.exists(p_path): 
                    continue
                    
                # 找到该目录下所有 .npy 文件（每个文件代表一个病人，如 4.npy）
                for img_file in os.scandir(p_path):
                    # 只找 .npy 文件，排除 slices 等子目录
                    if img_file.is_file() and img_file.name.endswith('.npy'):
                        all_patient_files.append(img_file.path)
        
        all_patient_files.sort()
        
        # 2. 根据 patient_indices 筛选病人（以病人为单位划分）
        if patient_indices is not None:
            all_patient_files = [all_patient_files[i] for i in patient_indices if i < len(all_patient_files)]
        
        # 3. 【关键修改】展开切片：遍历每个病人的 npy，统计切片数
        self.all_slices = []
        for patient_file in all_patient_files:
            vol = np.load(patient_file)
            num_slices = vol.shape[0] if vol.ndim == 3 else 1
            for s_idx in range(num_slices):
                self.all_slices.append((patient_file, s_idx))
        
        print(f"[{phase}] 病人数: {len(all_patient_files)}, 切片数: {len(self.all_slices)}")

    def __len__(self): 
        return len(self.all_slices)

    def __getitem__(self, idx):
        img_path, slice_idx = self.all_slices[idx]
        label_path = img_path.replace('_image', '_label')
        
        # 加载数据并提取对应切片
        image_vol = np.load(img_path)
        label_vol = np.load(label_path)
        
        image = image_vol[slice_idx] if image_vol.ndim == 3 else image_vol
        label = label_vol[slice_idx] if label_vol.ndim == 3 else label_vol
        
        # 标签重映射
        temp_label = np.zeros_like(label)
        temp_label[label == 2] = 1  # LV
        temp_label[label == 1] = 2  # RV
        label = temp_label

        # 归一化
        image = (image - image.min()) / (image.max() - image.min() + 1e-8)
        
        # 裁剪到目标尺寸
        img_pil = Image.fromarray((image * 255).astype(np.uint8))
        lab_pil = Image.fromarray(label.astype(np.uint8))
        
        img_pil = TF.center_crop(img_pil, [self.target_size, self.target_size])
        lab_pil = TF.center_crop(lab_pil, [self.target_size, self.target_size])
        
        return TF.to_tensor(img_pil), torch.from_numpy(np.array(lab_pil)).long()

## eval and plt

In [7]:
def calculate_dice(pred, target, n_classes=3):
    """
    计算多类别的 Dice 系数
    pred: [B, C, H, W] 模型的 logits
    target: [B, H, W] 真实标签
    """
    dice_scores = []
    # 将预测值转为类别索引 [B, H, W]
    pred_cls = torch.argmax(pred, dim=1)
    
    # 忽略背景 (class 0)，只计算前景类别 (1 和 2)
    for cl in range(1, n_classes):
        p = (pred_cls == cl).float()
        t = (target == cl).float()
        
        intersection = (p * t).sum()
        union = p.sum() + t.sum()
        
        if union == 0:
            dice = 1.0 # 如果预测和真实全为空，Dice 为 1
        else:
            dice = (2. * intersection) / (union + 1e-8)
        dice_scores.append(float(dice))
        
    return np.mean(dice_scores) # 返回前景类别的平均 Dice

In [8]:
def evaluate_model(model, loader, device):
    model.eval()
    dices = []
    if len(loader) == 0: return 0.0
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            outputs = model(imgs)
            dices.append(calculate_dice(outputs, masks))
    return np.mean(dices)

In [9]:
def visualize_result(model, dataset, device, num_samples=3):
    model.eval()
    fig, axes = plt.subplots(num_samples, 3, figsize=(12, 4 * num_samples))
    
    # 随机选几个样本
    random.seed(42)
    indices = random.sample(range(len(dataset)), num_samples)
    
    with torch.no_grad():
        for i, idx in enumerate(indices):
            img, mask = dataset[idx]
            img_in = img.unsqueeze(0).to(device)
            output = model(img_in)
            pred = torch.argmax(output, dim=1).cpu().squeeze(0)
            
            # 第一列：原图
            axes[i, 0].imshow(img.squeeze(), cmap='gray')
            axes[i, 0].set_title("Input Image")
            # 第二列：真实标签
            axes[i, 1].imshow(mask, cmap='tab10')
            axes[i, 1].set_title("Ground Truth")
            # 第三列：模型预测
            axes[i, 2].imshow(pred, cmap='tab10')
            axes[i, 2].set_title("Prediction")
            
            for ax in axes[i]: ax.axis('off')
            
    plt.tight_layout()
    plt.show()

# run

## supervised

In [ ]:
# def train(flag_plt=False):
#     # 1. 模型与优化器初始化
#     model = UNet(n_channels=1, n_classes=3).to(config.DEVICE)
#     optimizer = torch.optim.Adam(model.parameters(), lr=config.BASE_LR)
#     criterion = nn.CrossEntropyLoss()

#     # 2. 加载训练集并提取 5% 标注
#     full_train_dataset = Target_ACDC(phase='train')
#     num_labeled = int(len(full_train_dataset) * config.LABELED_RATIO)
#     indices = list(range(len(full_train_dataset)))
#     random.seed(42)
#     random.shuffle(indices)

    
#     labeled_subset = Subset(full_train_dataset, indices[:num_labeled])
#     train_loader = DataLoader(labeled_subset, batch_size=config.BATCH_SIZE, shuffle=True)

#     # 3. 加载验证集 (Val) 和 测试集 (Test)
#     # 这里的 phase 会自动去对应的目录找 slices
#     val_dataset = Target_ACDC(phase='val') 
#     val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False)
    
#     test_dataset = Target_ACDC(phase='test')
#     test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

#     print(f"训练({int(config.LABELED_RATIO*100)}%): {len(labeled_subset)} | 验证: {len(val_dataset)} | 测试: {len(test_dataset)}")

#     best_val_dice = 0.0

#     # 4. 训练循环
#     for epoch in range(config.EPOCHS):
#         model.train()
#         train_loss = 0
        
#         for imgs, masks in train_loader:
#             imgs, masks = imgs.to(config.DEVICE), masks.to(config.DEVICE)
            
#             optimizer.zero_grad()
#             outputs = model(imgs)
#             loss = criterion(outputs, masks)
#             loss.backward()
#             optimizer.step()
#             train_loss += loss.item()

#         # --- 每 5 个 Epoch 在验证集上跑一次 Dice ---
#         if (epoch + 1) % 5 == 0:
#             val_dice = evaluate_model(model, val_loader, config.DEVICE)
#             print(f"Epoch [{epoch+1}/{config.EPOCHS}] | Train Loss: {train_loss/len(train_loader):.4f} | Val Dice: {val_dice:.4f}")
            
#             # 保存表现最好的模型
#             if val_dice > best_val_dice:
#                 best_val_dice = val_dice
#                 torch.save(model.state_dict(), "supervised.pth")

#     # 5. 训练结束：在独立测试集上跑最终结果
#     model.load_state_dict(torch.load("supervised.pth")) # 加载验证集上表现最好的权重
#     test_dice = evaluate_model(model, test_loader, config.DEVICE)
#     print(f"Final Test Dice: {test_dice:.4f}")

#     # 6. 最终出图
#     if flag_plt:
#         visualize_result(model, test_dataset, config.DEVICE)
#     return test_dice

In [18]:
# def train_supervised(flag_plt=False):
#     """
#     纯有监督训练：仅使用 5% 的标注数据
#     集成代码 ② 的混合 Loss 和 Poly 学习率调整
#     """
#     # 1. 初始化模型与优化器
#     model = UNet(n_channels=1, n_classes=3).to(config.DEVICE)
    
#     # 按照代码 ② 使用权重衰减 (Weight Decay)
#     optimizer = torch.optim.Adam(model.parameters(), lr=config.BASE_LR, weight_decay=0.0001)
    
#     # 损失函数集成
#     criterion_ce = nn.CrossEntropyLoss()
#     criterion_dice = DiceLoss(n_classes=3)

#     # 2. 加载数据：仅使用标注集
#     full_train_dataset = Target_ACDC(phase='train')
#     total_samples = len(full_train_dataset)
#     num_labeled = int(total_samples * config.LABELED_RATIO)  # 仅使用 5%
    
#     indices = list(range(total_samples))
#     random.seed(42)
#     random.shuffle(indices)
    
#     labeled_subset = Subset(full_train_dataset, indices[:num_labeled])
#     labeled_loader = DataLoader(labeled_subset, batch_size=config.BATCH_SIZE, shuffle=True, drop_last=True)

#     # 加载验证集和测试集
#     val_dataset = Target_ACDC(phase='val')
#     val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False)
#     test_dataset = Target_ACDC(phase='test')
#     test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

#     print(f"有监督训练 | 标注数据量: {len(labeled_subset)} | 验证集: {len(val_dataset)}")

#     best_val_dice = 0.0
#     iter_num = 0
#     max_iterations = config.EPOCHS * len(labeled_loader)

#     # 3. 训练循环
#     for epoch in range(config.EPOCHS):
#         model.train()
#         train_loss_epoch = 0
        
#         for imgs, masks in labeled_loader:
#             imgs, masks = imgs.to(config.DEVICE), masks.to(config.DEVICE)
            
#             # 前向传播
#             outputs = model(imgs)
            
#             # --- 按照代码 ② 计算混合损失 ---
#             loss_ce = criterion_ce(outputs, masks)
#             loss_dice = criterion_dice(outputs, masks, softmax=True)
#             loss = 0.5 * (loss_ce + loss_dice)
            
#             optimizer.zero_grad()
#             loss.backward()
#             optimizer.step()
            
#             # --- 按照代码 ② 进行 Poly 学习率调整 ---
#             lr_ = config.BASE_LR * (1.0 - iter_num / max_iterations) ** 0.9
#             for param_group in optimizer.param_groups:
#                 param_group['lr'] = lr_
            
#             train_loss_epoch += loss.item()
#             iter_num += 1

#         # --- 每 5 个 Epoch 评估 ---
#         if (epoch + 1) % 5 == 0:
#             val_dice = evaluate_model(model, val_loader, config.DEVICE)
#             avg_loss = train_loss_epoch / len(labeled_loader)
#             print(f"Epoch [{epoch+1}/{config.EPOCHS}] | Loss: {avg_loss:.4f} | "
#                   f"Val Dice: {val_dice:.4f} | LR: {lr_:.6f}")
            
#             if val_dice > best_val_dice:
#                 best_val_dice = val_dice
#                 torch.save(model.state_dict(), "supervised_best.pth")

#     # 4. 测试与出图
#     model.load_state_dict(torch.load("supervised_best.pth"))
#     test_dice = evaluate_model(model, test_loader, config.DEVICE)
#     print(f"[Supervised] Final Test Dice: {test_dice:.4f}")

#     if flag_plt:
#         visualize_result(model, test_dataset, config.DEVICE)
        
#     return test_dice

In [10]:
def get_patient_id_from_path(filepath):
    """从文件路径提取病人编号，如 gp4_ED_0"""
    # 路径类似: /home/yzm/ACDC/gp4/ED/train_image/0.npy
    parts = filepath.replace('\\', '/').split('/')
    # 找到 gp 开头的目录
    gp_name = None
    frame_name = None
    patient_num = None
    for i, part in enumerate(parts):
        if part.startswith('gp'):
            gp_name = part
        if part in ['ED', 'ES']:
            frame_name = part
    # 文件名就是病人编号
    patient_num = os.path.splitext(os.path.basename(filepath))[0]
    if gp_name and frame_name and patient_num is not None:
        return f"{gp_name}_{frame_name}_{patient_num}"
    return filepath

def train_supervised(flag_plt=False):
    student_model = UNet(n_channels=1, n_classes=config.NUM_CLASSES).to(config.DEVICE)
    teacher_model = UNet(n_channels=1, n_classes=config.NUM_CLASSES).to(config.DEVICE)
    
    teacher_model.load_state_dict(student_model.state_dict())
    for param in teacher_model.parameters():
        param.detach_() # ② 使用 detach_ 取消梯度
    
    # ② 使用带有 momentum 的 SGD，但此处按 Jupyter 习惯保留 Adam 或改用带衰减的优化
    optimizer = torch.optim.Adam(student_model.parameters(), lr=config.BASE_LR, weight_decay=0.0001)
    
    ce_loss = nn.CrossEntropyLoss()
    dice_loss = DiceLoss(config.NUM_CLASSES)
    mse_loss = nn.MSELoss()

    # 数据准备：【关键修改】以病人为单位划分
    full_train_dataset = Target_ACDC(phase='train')
    
    # 先统计总病人数（而非切片数）- 直接在 train_image 目录找 npy，不进入 slices
    temp_patient_files = []
    group_path = [entry for entry in os.scandir(os.path.join(config.DATASET_DIR, "ACDC")) if entry.is_dir()]
    for gp in group_path:
        for frame_path in os.scandir(gp):
            if frame_path.name not in ['ED', 'ES']: continue
            # 【关键修改】直接在 train_image 目录找 npy，不进入 slices
            p_path = os.path.join(frame_path.path, 'train_image')
            if not os.path.exists(p_path): continue
            for img_file in os.scandir(p_path):
                # 只找 .npy 文件，排除 slices 等子目录
                if img_file.is_file() and img_file.name.endswith('.npy'):
                    temp_patient_files.append(img_file.path)
    temp_patient_files.sort()
    total_patients = len(temp_patient_files)
    
    # 以病人为单位计算数量
    num_labeled_patients = int(total_patients * config.LABELED_RATIO)
    num_unlabeled_patients = int(total_patients * config.UNLABELED_RATIO)
    
    # 打乱病人索引
    patient_indices = list(range(total_patients))
    random.seed(42)
    random.shuffle(patient_indices)
    
    # 划分病人索引
    labeled_patient_indices = patient_indices[:num_labeled_patients]
    unlabeled_patient_indices = patient_indices[num_labeled_patients:num_labeled_patients+num_unlabeled_patients]
    
    # 【新增】输出选取的病人编号
    labeled_patient_ids = [get_patient_id_from_path(temp_patient_files[i]) for i in labeled_patient_indices]
    unlabeled_patient_ids = [get_patient_id_from_path(temp_patient_files[i]) for i in unlabeled_patient_indices]
    print(f"\n=== 数据划分信息 ===")
    print(f"总病人数: {total_patients}")
    print(f"标注病人数 ({config.LABELED_RATIO*100:.0f}%): {num_labeled_patients}")
    print(f"标注病人编号: {labeled_patient_ids}")
    print(f"无标注病人数 ({config.UNLABELED_RATIO*100:.0f}%): {num_unlabeled_patients}")
    if len(unlabeled_patient_ids) <= 20:
        print(f"无标注病人编号: {unlabeled_patient_ids}")
    else:
        print(f"无标注病人编号 (前20个): {unlabeled_patient_ids[:20]} ...")
    print(f"====================\n")
    
    # 创建数据集（每个数据集内部会展开对应病人的所有切片）
    labeled_dataset = Target_ACDC(phase='train', patient_indices=labeled_patient_indices)
    unlabeled_dataset = Target_ACDC(phase='train', patient_indices=unlabeled_patient_indices)
    
    labeled_loader = DataLoader(labeled_dataset, batch_size=config.BATCH_SIZE//2, shuffle=True, drop_last=True)
    unlabeled_loader = DataLoader(unlabeled_dataset, batch_size=config.BATCH_SIZE//2, shuffle=True, drop_last=True)



    val_loader = DataLoader(Target_ACDC(phase='val'), batch_size=config.BATCH_SIZE, shuffle=False)
    test_dataset = Target_ACDC(phase='test')
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

    iter_num = 0
    max_iterations = config.EPOCHS * len(labeled_loader)
    best_val_dice = 0.0

    for epoch in range(config.EPOCHS):
        student_model.train()
        teacher_model.train()
        
        unlabeled_iter = iter(unlabeled_loader)

        # 训练循环内：用纯监督分支
        for imgs_l, masks_l in labeled_loader:
            imgs_l, masks_l = imgs_l.to(config.DEVICE), masks_l.to(config.DEVICE)

            outputs_l = student_model(imgs_l)
            loss_ce = ce_loss(outputs_l, masks_l)
            loss_dice = dice_loss(outputs_l, masks_l, softmax=True)
            supervised_loss = 0.5 * (loss_ce + loss_dice)

            # 直接不计算一致性损失，也不使用 unlabeled 前向
            loss = supervised_loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # 可选：注释掉 EMA 更新，避免 teacher 影响
            # update_ema_variables(student_model, teacher_model, config.EMA_ALPHA, iter_num)

            # --- ④ Poly 学习率调整 ---
            lr_ = config.BASE_LR * (1.0 - iter_num / max_iterations) ** 0.9
            for param_group in optimizer.param_groups:
                param_group['lr'] = lr_

            iter_num += 1

        # 评估逻辑 (保留 ① 的每 5 Epoch 评估和打印)
        if (epoch + 1) % 5 == 0:
            val_dice = evaluate_model(student_model, val_loader, config.DEVICE)
            print(f"Epoch [{epoch+1}/{config.EPOCHS}] | Iter {iter_num} | "
                  f"Loss: {loss.item():.4f} | Val Dice: {val_dice:.4f} | LR: {lr_:.6f}")
            
            if val_dice > best_val_dice:
                best_val_dice = val_dice
                torch.save(student_model.state_dict(), "semi_supervised_best.pth")

    # 训练结束
    student_model.load_state_dict(torch.load("semi_supervised_best.pth"))
    test_dice = evaluate_model(student_model, test_loader, config.DEVICE)
    print(f"[Finished] Final Test Dice: {test_dice:.4f}")

    if flag_plt:
        visualize_result(student_model, test_dataset, config.DEVICE)
        
    return test_dice

In [11]:
dice_scores = []
config.EPOCHS = 150
for i in range(10):
    dice = train_supervised() 
    dice_scores.append(dice)

[train] 病人数: 140, 切片数: 1276

=== 数据划分信息 ===
总病人数: 140
标注病人数 (10%): 14
标注病人编号: ['gp5_ES_13', 'gp4_ED_0', 'gp5_ES_8', 'gp1_ED_4', 'gp3_ED_2', 'gp1_ES_14', 'gp3_ES_15', 'gp4_ES_11', 'gp2_ES_17', 'gp4_ES_8', 'gp1_ED_13', 'gp5_ES_17', 'gp4_ES_12', 'gp5_ES_11']
无标注病人数 (90%): 126
无标注病人编号 (前20个): ['gp5_ED_9', 'gp4_ES_10', 'gp5_ED_14', 'gp2_ED_13', 'gp1_ES_16', 'gp3_ED_5', 'gp1_ES_2', 'gp4_ED_12', 'gp5_ES_14', 'gp2_ED_19', 'gp4_ED_18', 'gp2_ES_4', 'gp2_ED_5', 'gp2_ED_11', 'gp5_ED_5', 'gp2_ES_19', 'gp2_ED_14', 'gp5_ED_19', 'gp3_ES_13', 'gp4_ES_0'] ...

[train] 病人数: 14, 切片数: 123
[train] 病人数: 126, 切片数: 1153
[val] 病人数: 30, 切片数: 282
[test] 病人数: 30, 切片数: 283
Epoch [5/150] | Iter 150 | Loss: 0.2007 | Val Dice: 0.0000 | LR: 0.009702
Epoch [10/150] | Iter 300 | Loss: 0.3459 | Val Dice: 0.1300 | LR: 0.009400
Epoch [15/150] | Iter 450 | Loss: 0.2415 | Val Dice: 0.2646 | LR: 0.009097
Epoch [20/150] | Iter 600 | Loss: 0.1426 | Val Dice: 0.0120 | LR: 0.008794
Epoch [25/150] | Iter 750 | Loss: 0.1504 | Val Di

In [12]:
for idx, score in enumerate(dice_scores):
    print(f"Run {idx+1}: Test Dice = {score}")
print(f"Average Dices: {np.mean(dice_scores)}")

Run 1: Test Dice = 0.6998666895186535
Run 2: Test Dice = 0.7369265121855368
Run 3: Test Dice = 0.6755461872945286
Run 4: Test Dice = 0.6760765230398322
Run 5: Test Dice = 0.6295526224474814
Run 6: Test Dice = 0.6865401105780826
Run 7: Test Dice = 0.6779196482158328
Run 8: Test Dice = 0.6611809471997271
Run 9: Test Dice = 0.7250733165886556
Run 10: Test Dice = 0.6694805004811671
Average Dices: 0.6838163057549498


## semi-

In [35]:
# import copy

# def update_ema_variables(model, ema_model, alpha):
#     """
#     使用指数移动平均 (EMA) 更新教师模型的参数
#     """
#     with torch.no_grad():
#         for ema_param, param in zip(ema_model.parameters(), model.parameters()):
#             ema_param.data.mul_(alpha).add_(param.data, alpha=1 - alpha)

# def get_current_consistency_weight(epoch, max_epochs, max_weight):
#     """
#     一致性权重 ramp-up：从 0 逐渐增加到 max_weight
#     使用 sigmoid 型 ramp-up 曲线
#     """
#     # 在前 80% 的 epoch 进行 ramp-up
#     rampup_length = int(max_epochs * 0.8)
#     if epoch < rampup_length:
#         p = epoch / rampup_length
#         return max_weight * np.exp(-5 * (1 - p) ** 2)
#     return max_weight

# def train_semi_supervised(flag_plt=False):
#     """
#     半监督训练：Mean Teacher 方法
#     使用 5% 标注数据 + 45% 无标注数据
#     """
#     # 1. 初始化学生模型和教师模型
#     student_model = UNet(n_channels=1, n_classes=3).to(config.DEVICE)
#     teacher_model = UNet(n_channels=1, n_classes=3).to(config.DEVICE)
    
#     # 教师模型初始化为学生模型的副本，且不需要梯度
#     teacher_model.load_state_dict(student_model.state_dict())
#     for param in teacher_model.parameters():
#         param.requires_grad = False
    
#     optimizer = torch.optim.Adam(student_model.parameters(), lr=config.LEARNING_RATE)
#     criterion_sup = nn.CrossEntropyLoss()  # 监督损失
#     criterion_con = nn.MSELoss()           # 一致性损失 (soft label)

#     # 2. 加载训练集并划分标注/无标注数据
#     full_train_dataset = Target_ACDC(phase='train')
#     total_samples = len(full_train_dataset)
#     num_labeled = int(total_samples * config.LABELED_RATIO)      # 5%
#     num_unlabeled = int(total_samples * config.UNLABELED_RATIO)  # 45%
    
#     indices = list(range(total_samples))
#     random.seed(42)
#     random.shuffle(indices)

    
#     labeled_indices = indices[:num_labeled]
#     unlabeled_indices = indices[num_labeled:num_labeled + num_unlabeled]
    
#     labeled_subset = Subset(full_train_dataset, labeled_indices)
#     unlabeled_subset = Subset(full_train_dataset, unlabeled_indices)
    
#     labeled_loader = DataLoader(labeled_subset, batch_size=config.BATCH_SIZE, shuffle=True, drop_last=True)
#     unlabeled_loader = DataLoader(unlabeled_subset, batch_size=config.BATCH_SIZE, shuffle=True, drop_last=True)

#     # 3. 加载验证集和测试集
#     val_dataset = Target_ACDC(phase='val')
#     val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False)
    
#     test_dataset = Target_ACDC(phase='test')
#     test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

#     print(f"半监督训练 | 标注({int(config.LABELED_RATIO*100)}%): {len(labeled_subset)} | "
#         f"无标注({int(config.UNLABELED_RATIO*100)}%): {len(unlabeled_subset)} | "
#         f"验证: {len(val_dataset)} | 测试: {len(test_dataset)}")

#     best_val_dice = 0.0

#     # 4. 训练循环
#     for epoch in range(config.EPOCHS):
#         student_model.train()
#         teacher_model.train()  # 教师模型也设为 train 模式（用于 BN 统计）
        
#         train_loss_sup = 0
#         train_loss_con = 0
        
#         # 获取当前 epoch 的一致性权重
#         consistency_weight = get_current_consistency_weight(
#             epoch, config.EPOCHS, config.CONSISTENCY_WEIGHT
#         )
        
#         # 创建无标注数据的迭代器
#         unlabeled_iter = iter(unlabeled_loader)
        
#         for labeled_batch in labeled_loader:
#             imgs_l, masks_l = labeled_batch
#             imgs_l, masks_l = imgs_l.to(config.DEVICE), masks_l.to(config.DEVICE)
            
#             # --- 监督损失 (标注数据) ---
#             optimizer.zero_grad()
#             outputs_l = student_model(imgs_l)
#             loss_sup = criterion_sup(outputs_l, masks_l)
            
#             # --- 一致性损失 (无标注数据) ---
#             loss_con = torch.tensor(0.0).to(config.DEVICE)
#             try:
#                 imgs_u, _ = next(unlabeled_iter)  # 忽略无标注数据的 label
#             except StopIteration:
#                 unlabeled_iter = iter(unlabeled_loader)
#                 imgs_u, _ = next(unlabeled_iter)
            
#             imgs_u = imgs_u.to(config.DEVICE)
            
#             # 学生模型预测
#             student_outputs_u = student_model(imgs_u)
#             student_probs_u = F.softmax(student_outputs_u, dim=1)
            
#             # 教师模型预测 (无梯度)
#             with torch.no_grad():
#                 teacher_outputs_u = teacher_model(imgs_u)
#                 teacher_probs_u = F.softmax(teacher_outputs_u, dim=1)
            
#             # 一致性损失：学生预测与教师预测的 MSE
#             loss_con = criterion_con(student_probs_u, teacher_probs_u)
            
#             # --- 总损失 ---
#             total_loss = loss_sup + consistency_weight * loss_con
#             total_loss.backward()
#             optimizer.step()
            
#             # --- 更新教师模型 (EMA) ---
#             update_ema_variables(student_model, teacher_model, config.EMA_ALPHA)
            
#             train_loss_sup += loss_sup.item()
#             train_loss_con += loss_con.item()

#         # --- 每 5 个 Epoch 在验证集上评估 ---
#         if (epoch + 1) % 5 == 0:
#             val_dice = evaluate_model(student_model, val_loader, config.DEVICE)
#             avg_sup = train_loss_sup / len(labeled_loader)
#             avg_con = train_loss_con / len(labeled_loader)
#             print(f"Epoch [{epoch+1}/{config.EPOCHS}] | "
#                 f"Sup Loss: {avg_sup:.4f} | Con Loss: {avg_con:.4f} | "
#                 f"Con Weight: {consistency_weight:.4f} | Val Dice: {val_dice:.4f}")
            
#             # 保存表现最好的模型
#             if val_dice > best_val_dice:
#                 best_val_dice = val_dice
#                 torch.save(student_model.state_dict(), "semi_supervised.pth")

#     # 5. 训练结束：在独立测试集上跑最终结果
#     student_model.load_state_dict(torch.load("semi_supervised.pth"))
#     test_dice = evaluate_model(student_model, test_loader, config.DEVICE)
#     print(f"[Semi-Supervised] Final Test Dice: {test_dice:.4f}")

#     # 6. 最终出图
#     if flag_plt:
#         visualize_result(student_model, test_dataset, config.DEVICE)
        
#     return test_dice

## semi2

In [13]:
def update_ema_variables(model, ema_model, alpha, global_step):
    # ② 的动态 alpha 逻辑：初期更新快，后期稳定
    alpha = min(1 - 1 / (global_step + 1), alpha)
    with torch.no_grad():
        for ema_param, param in zip(ema_model.parameters(), model.parameters()):
            ema_param.data.mul_(alpha).add_(param.data, alpha=1 - alpha)

def get_current_consistency_weight(epoch):
    # ② 的 sigmoid 预热逻辑
    import math
    if epoch < config.CONSISTENCY_RAMPUP:
        p = max(0.0, float(epoch) / config.CONSISTENCY_RAMPUP)
        return config.CONSISTENCY_WEIGHT * math.exp(-5.0 * (1.0 - p)**2)
    else:
        return config.CONSISTENCY_WEIGHT

# ==========================================
# 4. 训练逻辑 (按 ② 修改)
# ==========================================
def train_semi_supervised(flag_plt=False):
    student_model = UNet(n_channels=1, n_classes=config.NUM_CLASSES).to(config.DEVICE)
    teacher_model = UNet(n_channels=1, n_classes=config.NUM_CLASSES).to(config.DEVICE)
    
    teacher_model.load_state_dict(student_model.state_dict())
    for param in teacher_model.parameters():
        param.detach_() # ② 使用 detach_ 取消梯度
    
    # ② 使用带有 momentum 的 SGD，但此处按 Jupyter 习惯保留 Adam 或改用带衰减的优化
    optimizer = torch.optim.Adam(student_model.parameters(), lr=config.BASE_LR, weight_decay=0.0001)
    
    ce_loss = nn.CrossEntropyLoss()
    dice_loss = DiceLoss(config.NUM_CLASSES)
    mse_loss = nn.MSELoss()

    # 数据准备：【关键修改】以病人为单位划分
    full_train_dataset = Target_ACDC(phase='train')
    
    # 先统计总病人数（而非切片数）- 直接在 train_image 目录找 npy，不进入 slices
    temp_patient_files = []
    group_path = [entry for entry in os.scandir(os.path.join(config.DATASET_DIR, "ACDC")) if entry.is_dir()]
    for gp in group_path:
        for frame_path in os.scandir(gp):
            if frame_path.name not in ['ED', 'ES']: continue
            # 【关键修改】直接在 train_image 目录找 npy，不进入 slices
            p_path = os.path.join(frame_path.path, 'train_image')
            if not os.path.exists(p_path): continue
            for img_file in os.scandir(p_path):
                # 只找 .npy 文件，排除 slices 等子目录
                if img_file.is_file() and img_file.name.endswith('.npy'):
                    temp_patient_files.append(img_file.path)
    temp_patient_files.sort()
    total_patients = len(temp_patient_files)
    
    # 以病人为单位计算数量
    num_labeled_patients = int(total_patients * config.LABELED_RATIO)
    num_unlabeled_patients = int(total_patients * config.UNLABELED_RATIO)
    
    # 打乱病人索引
    patient_indices = list(range(total_patients))
    random.seed(42)
    random.shuffle(patient_indices)
    
    # 划分病人索引
    labeled_patient_indices = patient_indices[:num_labeled_patients]
    unlabeled_patient_indices = patient_indices[num_labeled_patients:num_labeled_patients+num_unlabeled_patients]
    
    # 【新增】输出选取的病人编号
    labeled_patient_ids = [get_patient_id_from_path(temp_patient_files[i]) for i in labeled_patient_indices]
    unlabeled_patient_ids = [get_patient_id_from_path(temp_patient_files[i]) for i in unlabeled_patient_indices]
    print(f"\n=== 数据划分信息 ===")
    print(f"总病人数: {total_patients}")
    print(f"标注病人数 ({config.LABELED_RATIO*100:.0f}%): {num_labeled_patients}")
    print(f"标注病人编号: {labeled_patient_ids}")
    print(f"无标注病人数 ({config.UNLABELED_RATIO*100:.0f}%): {num_unlabeled_patients}")
    if len(unlabeled_patient_ids) <= 20:
        print(f"无标注病人编号: {unlabeled_patient_ids}")
    else:
        print(f"无标注病人编号 (前20个): {unlabeled_patient_ids[:20]} ...")
    print(f"====================\n")
    
    # 创建数据集（每个数据集内部会展开对应病人的所有切片）
    labeled_dataset = Target_ACDC(phase='train', patient_indices=labeled_patient_indices)
    unlabeled_dataset = Target_ACDC(phase='train', patient_indices=unlabeled_patient_indices)
    
    labeled_loader = DataLoader(labeled_dataset, batch_size=config.BATCH_SIZE//2, shuffle=True, drop_last=True)
    unlabeled_loader = DataLoader(unlabeled_dataset, batch_size=config.BATCH_SIZE//2, shuffle=True, drop_last=True)

    val_loader = DataLoader(Target_ACDC(phase='val'), batch_size=config.BATCH_SIZE, shuffle=False)
    test_dataset = Target_ACDC(phase='test')
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

    iter_num = 0
    max_iterations = config.EPOCHS * len(labeled_loader)
    best_val_dice = 0.0

    for epoch in range(config.EPOCHS):
        student_model.train()
        teacher_model.train()
        
        unlabeled_iter = iter(unlabeled_loader)
        
        for imgs_l, masks_l in labeled_loader:
            try:
                imgs_u, _ = next(unlabeled_iter)
            except StopIteration:
                unlabeled_iter = iter(unlabeled_loader)
                imgs_u, _ = next(unlabeled_iter)

            imgs_l, masks_l = imgs_l.to(config.DEVICE), masks_l.to(config.DEVICE)
            imgs_u = imgs_u.to(config.DEVICE)

            # --- ② 的输入扰动：为教师模型输入添加噪声 ---
            noise = torch.clamp(torch.randn_like(imgs_u) * 0.1, -0.2, 0.2)
            ema_inputs = imgs_u + noise
            
            #ema_inputs = imgs_u # 不加噪声，保持原图输入教师模型 

            # --- 学生模型前向传播 ---
            # 拼接有标签和无标签数据一起过学生模型 (模拟 ② 的混合 Batch)
            all_inputs = torch.cat([imgs_l, imgs_u], dim=0)
            all_outputs = student_model(all_inputs)
            outputs_l = all_outputs[:config.BATCH_SIZE//2]
            outputs_u = all_outputs[config.BATCH_SIZE//2:]
            outputs_u_soft = torch.softmax(outputs_u, dim=1)

            # --- 教师模型前向传播 (无梯度) ---
            with torch.no_grad():
                ema_output = teacher_model(ema_inputs)
                ema_output_soft = torch.softmax(ema_output, dim=1)

            # --- 1. 监督损失 (CE + Dice) ---
            loss_ce = ce_loss(outputs_l, masks_l)
            loss_dice = dice_loss(outputs_l, masks_l, softmax=True)
            supervised_loss = 0.5 * (loss_ce + loss_dice)

            # --- 2. 一致性损失 (带预热门控) ---
            consistency_weight = get_current_consistency_weight(iter_num // 150)
            if iter_num < 1000: # ② 的预热门控
                consistency_loss = torch.tensor(0.0).to(config.DEVICE)
            else:
                consistency_loss = mse_loss(outputs_u_soft, ema_output_soft)

            loss = supervised_loss + consistency_weight * consistency_loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # --- ③ 动态更新教师模型 (EMA) ---
            update_ema_variables(student_model, teacher_model, config.EMA_ALPHA, iter_num)

            # --- ④ Poly 学习率调整 ---
            lr_ = config.BASE_LR * (1.0 - iter_num / max_iterations) ** 0.9
            for param_group in optimizer.param_groups:
                param_group['lr'] = lr_

            iter_num += 1

        # 评估逻辑 (保留 ① 的每 5 Epoch 评估和打印)
        if (epoch + 1) % 5 == 0:
            val_dice = evaluate_model(student_model, val_loader, config.DEVICE)
            print(f"Epoch [{epoch+1}/{config.EPOCHS}] | Iter {iter_num} | "
                  f"Loss: {loss.item():.4f} | Val Dice: {val_dice:.4f} | LR: {lr_:.6f}")
            
            if val_dice > best_val_dice:
                best_val_dice = val_dice
                torch.save(student_model.state_dict(), "semi_supervised_best.pth")

    # 训练结束
    student_model.load_state_dict(torch.load("semi_supervised_best.pth"))
    test_dice = evaluate_model(student_model, test_loader, config.DEVICE)
    print(f"[Finished] Final Test Dice: {test_dice:.4f}")

    if flag_plt:
        visualize_result(student_model, test_dataset, config.DEVICE)
        
    return test_dice

In [14]:
dice_scores = []
config.UNLABELED_RATIO = 0.40
config.EPOCHS = 150
for i in range(10):
    dice = train_semi_supervised() 
    dice_scores.append(dice)

[train] 病人数: 140, 切片数: 1276

=== 数据划分信息 ===
总病人数: 140
标注病人数 (10%): 14
标注病人编号: ['gp5_ES_13', 'gp4_ED_0', 'gp5_ES_8', 'gp1_ED_4', 'gp3_ED_2', 'gp1_ES_14', 'gp3_ES_15', 'gp4_ES_11', 'gp2_ES_17', 'gp4_ES_8', 'gp1_ED_13', 'gp5_ES_17', 'gp4_ES_12', 'gp5_ES_11']
无标注病人数 (40%): 56
无标注病人编号 (前20个): ['gp5_ED_9', 'gp4_ES_10', 'gp5_ED_14', 'gp2_ED_13', 'gp1_ES_16', 'gp3_ED_5', 'gp1_ES_2', 'gp4_ED_12', 'gp5_ES_14', 'gp2_ED_19', 'gp4_ED_18', 'gp2_ES_4', 'gp2_ED_5', 'gp2_ED_11', 'gp5_ED_5', 'gp2_ES_19', 'gp2_ED_14', 'gp5_ED_19', 'gp3_ES_13', 'gp4_ES_0'] ...

[train] 病人数: 14, 切片数: 123
[train] 病人数: 56, 切片数: 513
[val] 病人数: 30, 切片数: 282
[test] 病人数: 30, 切片数: 283
Epoch [5/150] | Iter 150 | Loss: 0.1791 | Val Dice: 0.0927 | LR: 0.009702
Epoch [10/150] | Iter 300 | Loss: 0.2405 | Val Dice: 0.0000 | LR: 0.009400
Epoch [15/150] | Iter 450 | Loss: 0.1319 | Val Dice: 0.2007 | LR: 0.009097
Epoch [20/150] | Iter 600 | Loss: 0.1991 | Val Dice: 0.5738 | LR: 0.008794
Epoch [25/150] | Iter 750 | Loss: 0.0944 | Val Dice:

In [15]:
for idx, score in enumerate(dice_scores):
    print(f"Run {idx+1}: Test Dice = {score}")
print(f"Average Dices: {np.mean(dice_scores)}")

Run 1: Test Dice = 0.6868304015637471
Run 2: Test Dice = 0.7197212349235116
Run 3: Test Dice = 0.763437325148037
Run 4: Test Dice = 0.6649310369893593
Run 5: Test Dice = 0.6620666001669984
Run 6: Test Dice = 0.7134257858888947
Run 7: Test Dice = 0.6763645405510882
Run 8: Test Dice = 0.7559171834624909
Run 9: Test Dice = 0.649828620156678
Run 10: Test Dice = 0.681541156171831
Average Dices: 0.6974063885022637


In [16]:
dice_scores = []
config.UNLABELED_RATIO = 0.90
config.EPOCHS = 150
for i in range(10):
    dice = train_semi_supervised() 
    dice_scores.append(dice)

[train] 病人数: 140, 切片数: 1276

=== 数据划分信息 ===
总病人数: 140
标注病人数 (10%): 14
标注病人编号: ['gp5_ES_13', 'gp4_ED_0', 'gp5_ES_8', 'gp1_ED_4', 'gp3_ED_2', 'gp1_ES_14', 'gp3_ES_15', 'gp4_ES_11', 'gp2_ES_17', 'gp4_ES_8', 'gp1_ED_13', 'gp5_ES_17', 'gp4_ES_12', 'gp5_ES_11']
无标注病人数 (90%): 126
无标注病人编号 (前20个): ['gp5_ED_9', 'gp4_ES_10', 'gp5_ED_14', 'gp2_ED_13', 'gp1_ES_16', 'gp3_ED_5', 'gp1_ES_2', 'gp4_ED_12', 'gp5_ES_14', 'gp2_ED_19', 'gp4_ED_18', 'gp2_ES_4', 'gp2_ED_5', 'gp2_ED_11', 'gp5_ED_5', 'gp2_ES_19', 'gp2_ED_14', 'gp5_ED_19', 'gp3_ES_13', 'gp4_ES_0'] ...

[train] 病人数: 14, 切片数: 123
[train] 病人数: 126, 切片数: 1153
[val] 病人数: 30, 切片数: 282
[test] 病人数: 30, 切片数: 283
Epoch [5/150] | Iter 150 | Loss: 0.3649 | Val Dice: 0.2486 | LR: 0.009702
Epoch [10/150] | Iter 300 | Loss: 0.2137 | Val Dice: 0.0000 | LR: 0.009400
Epoch [15/150] | Iter 450 | Loss: 0.1919 | Val Dice: 0.0172 | LR: 0.009097
Epoch [20/150] | Iter 600 | Loss: 0.1207 | Val Dice: 0.5370 | LR: 0.008794
Epoch [25/150] | Iter 750 | Loss: 0.1459 | Val Di

In [17]:
for idx, score in enumerate(dice_scores):
    print(f"Run {idx+1}: Test Dice = {score}")
print(f"Average Dices: {np.mean(dice_scores)}")

Run 1: Test Dice = 0.6731332325353534
Run 2: Test Dice = 0.6559382297403867
Run 3: Test Dice = 0.6867068429698002
Run 4: Test Dice = 0.7401474057515761
Run 5: Test Dice = 0.6997187343942719
Run 6: Test Dice = 0.725234691282346
Run 7: Test Dice = 0.6814751777083814
Run 8: Test Dice = 0.6551854594317742
Run 9: Test Dice = 0.7532933275511509
Run 10: Test Dice = 0.7133284586229997
Average Dices: 0.6984161559988041


# eof

In [ ]:
dice_scores = []
config.UNLABELED_RATIO = 0.95
config.EPOCHS = 50
for i in range(10):
    dice = train_semi_supervised() 
    dice_scores.append(dice)

In [ ]:
for idx, score in enumerate(dice_scores):
    print(f"Run {idx+1}: Test Dice = {score}")
print(f"Average Dices: {np.mean(dice_scores)}")

In [ ]:
dice_scores = []
config.UNLABELED_RATIO = 0.45
config.EPOCHS = 150
for i in range(10):
    dice = train_semi_supervised() 
    dice_scores.append(dice)

In [ ]:
for idx, score in enumerate(dice_scores):
    print(f"Run {idx+1}: Test Dice = {score}")
print(f"Average Dices: {np.mean(dice_scores)}")